# Aula 4

# Redes Neurais Convolucionais com Keras

### Eduardo Lobo Lustosa Cabral


## 1. Objetivo

Apresentar como configurar, compilar e treinar uma RNA convolucional com o Keras.

Apresentar como visualizar as saídas das camadas convolucionais de uma RNA.

Apresentar o que uma RNA convolucional realiza nas suas camadas.

## 2. Conjunto de dados

Para exemplificar a configuração e treinamento de uma rede convolucional no Keras será resolvido um problema de classificação multiclasse usando o conjunto de dados MINIST de dígitos.

### 2.1 Importação das principais bibliotecas

In [ ]:
import tensorflow as tf
import matplotlib.pyplot as plt
import numpy as np

print('TensorFlow version:', tf.__version__)

### 2.2 Importação do conjunto de dados MNIST de dígitos

In [ ]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

print('Dimensão dos dados de entrada de treinamento =', x_train.shape)
print('Dimensão dos dados de entrada de teste =', x_test.shape)
print('Dimensão dos dados de saída de treinamento =', y_train.shape)
print('Dimensão dos dados de saída de teste =', y_test.shape)

In [ ]:
# Exemplo de uma imagem
index = 3
plt.imshow(x_train[index], cmap='gray', vmin=0, vmax=255)
print ("É uma figura do número " + str(y_train[index]))

### 2.3 Inclusão do eixo de cores e normalização das imagens

In [ ]:
# Determina número de exemplos
m = x_train.shape[0]

# Inclui eixo de cores nas imagens
x_train = np.expand_dims(x_train, 3)
x_test = np.expand_dims(x_test, 3)

# Determina dimensões das imagens
img_dim = x_train.shape[1:]
print('Dimensão das imagens =', img_dim)

# Normaliza imagens
x_train = x_train/255.
x_test = x_test/255.

print('Dimensão dos dados de entrada de treinamento =', x_train.shape)
print('Dimensão dos dados de entrada de teste =', x_test.shape)

In [ ]:
print(np.min(x_train), np.max(x_train))

### 2.4 Codificação one-hot das saídas

In [ ]:
print('Classes das imagens:', y_train[:10])

In [ ]:
# Importa classe de utilidades do Keras
from tensorflow.keras.utils import to_categorical

# Transformação das classes de números reais para vetores
y_train_hot = to_categorical(y_train)
y_test_hot = to_categorical(y_test)

print('Dimensão dos dados de saída do conjunto de treinamento: ', y_train_hot.shape)
print('Dimensão dos dados de saída do conjunto de teste: ', y_test_hot.shape)
print('Classe numérica:', y_train[:5])
print('Classe como vetor one-hot:\n', y_train_hot[:5])

## 3.Configuração de uma RNA convolucional

A configuração de uma RNA convolucional usando o Keras segue os mesmos princípios usados para as redes com camadas densas.


### 3.1 Rede LeNet-5

Como exemplo a rede LeNet-5, que é usada para classificação multi-classe, possui a seguinte arquitetura:

- Dimensão das imagens de entrada: 32x32x3;


- Camada convolucional: $f^{[1]} = 5$, $p^{[1]} = 0$, $s^{[1]} = 1$, $n_c^{[1]} = 8$, ReLu;


- Camada de “max-pooling”: $f^{[2]} = 2$, $s^{[2]} = 2$;


- Camada convolucional: $f^{[3]} = 5$, $p^{[3]} = 0$, $s^{[3]} = 1$, $n_c^{[3]} = 16$, ReLu;


- Camada de “max-pooling”: $f^{[4]} = 2$, $s^{[4]} = 2$;


- Camada de “flattening”;


- Camada densa: $n^{[5]} = 120$, ReLu;


- Camada densa: $n^{[6]} = 84$, ReLu;


- Camada softmax: $n^{[7]} = 10$, softmax.

<br>

**Arquitetura da RN LeNet-5**

<br>
<img src="LeNet.png">


O código a seguir apresenta como configurar as duas camadas convolucionais e as duas camadas de “pooling” da rede LeNet-5.

In [ ]:
from tensorflow.keras import layers
from tensorflow.keras import models

rna = models.Sequential()
rna.add(layers.Conv2D(8, (5, 5), strides=1, padding='valid', activation='relu', input_shape=img_dim))
rna.add(layers.MaxPooling2D((2, 2)))
rna.add(layers.Conv2D(16, (5, 5), strides=1, padding='valid', activation='relu'))
rna.add(layers.MaxPooling2D((2, 2)))

rna.summary()

Npar = 5 * 5 * 8 * 16 + 16 = 3216

- Note que o número de filtros (número de canais) é o primeiro argumento passado na adição da camada convolucional.


- Na camada de “pooling” o primeiro valor é a dimensão da janela e o segundo é o “stride”.


- O padrão das camadas convolucionais no Keras é $s = 1$ e $p = 0$ (“valid”), assim, se esse for o caso não precisaria incluir essas informações.

Falta adicionar na RNA a camada de “flattening” e as camadas densas. O código a seguir mostra como fazer essa etapa.

In [ ]:
rna.add(layers.Flatten())
rna.add(layers.Dense(120, activation='relu'))
rna.add(layers.Dense(84, activation='relu'))
rna.add(layers.Dense(10, activation='softmax'))

rna.summary()

- O redimensionamento da saída da última camada convolucional para transformar o tensor 3D em um vetor é realizada pela camada `Flatten`.

Um esquema da rede é realizado abaixo.

In [ ]:
# Importa função para fazer gráfico de RNAs
from tensorflow.keras.utils import plot_model

# Cria um arquivo com o esquema da VGG16
plot_model(rna, to_file='rna.png', show_shapes=True)

## 4. Compilação e treinamento

A compilação e treinamento de uma RNA convolucional no Keras é realizada exatamente como nas redes com somente camadas densas.

Na célula abaixo é feita a compilação da RN usando o método de otimização RMSprop, com todos os parâmetros padrão.

In [ ]:
rna.compile(optimizer='rmsprop', loss='categorical_crossentropy', metrics=['accuracy'])

Na célula abaixo é realizado o treinamento da RN com 20 épocas de treinamento.

In [ ]:
history = rna.fit(x_train, y_train_hot, epochs=20, verbose=1, batch_size=1000, validation_data=(x_test, y_test_hot))

In [ ]:
# Salva treinamento na variável history para visualização
history_dict = history.history

# Salva custos, métricas e epocas em vetores
custo = history_dict['loss']
acc = history_dict['accuracy']
val_custo = history_dict['val_loss']
val_acc = history_dict['val_accuracy']

# Cria vetor de épocas
epocas = range(1, len(custo) + 1)

# Gráfico dos valores de custo
plt.plot(epocas, custo, 'b', label='Custo treinamento')
plt.plot(epocas, val_custo, 'r', label='Custo validação')
plt.xlabel('Épocas')
plt.ylabel('Custo')
plt.legend()
plt.grid()
plt.show()

# Gráfico dos valores da métrica
plt.plot(epocas, acc, 'b', label='Exatidao treinamento')
plt.plot(epocas, val_acc, 'r', label='Exatidao validação')
plt.title('Valor da métrica')
plt.xlabel('Épocas')
plt.ylabel('Exatidao')
plt.legend()
plt.grid()
plt.show()

## 5. Avaliação e teste da RN treinada


### 5.1 Avaliação do desempenho

In [ ]:
# Avaliação da RN para os dados de treinamento e teste
custo_e_metricas_train = rna.evaluate(x_train, y_train_hot)
custo_e_metricas_test = rna.evaluate(x_test, y_test_hot)

### 5.2 Avaliação das previsões

Note que a previsão da RNA é um vetor de 10 elementos com as probabilidades da imagem mostrar os 10 dígitos. Para detereminar a classe prevista deve-se transformar esse vetor em um número inteiro de 0 a 9, que representa o dígito sendo mostrado.

In [ ]:
# Calculo das previsões da RNA
y_pred = rna.predict(x_test)

# Seleção da imagem
index = 2000
print(y_pred[index])

# Cálculo das classes previstas
classe = np.argmax(y_pred, axis=1)

# Exemplo de uma imagem dos dados de teste
plt.imshow(np.squeeze(x_test[index]), cmap='gray')
plt.show()
print ("classe prevista = " + str(np.squeeze(classe[index])))
print ("classe real = " + str(np.squeeze(y_test[index])))

Cálculo das previsões da RNA para as imagens dos dados de teste e verificação se algumas dessas previsões estão corretas.

In [ ]:
# Calculo das previsões da RNA
y_test_prev = rna.predict(x_test)

# Cálculo das classes previstas
classe = np.argmax(y_test_prev, axis=1)

# Gráfico das classes reais e previstas (100 primeiros exemplos)
plt.figure(figsize=(16, 6))
plt.plot(y_test[:250], 'ro', label='Classes reais')
plt.plot(classe[:250], 'bo', label='Classes previstas')
plt.title('Classes reais e previstas')
plt.xlabel('Exemplos')
plt.ylabel('Classes')
plt.legend()
plt.grid()
plt.show()

## 6. Visualização do aprendizado

A visualização das saídas das camadas convolucionais de uma RNA é importante para entender o que de fato a rede aprendeu e o que está fazendo para processar os dados.

Existem várias formas de visualização e interpretação dos resultados de uma RNA convolucional. As três formas mais utilizadas são:

- Visualização das saídas das camadas intermediárias da rede $\to$ útil para entender como as sucessivas camadas convolucionais transformam suas entradas e entender o que realiza cada filtro da camada;


- Visualização dos filtros $\to$ útil para entender cada padrão que a rede é capaz de detectar e conhecer os filtros que a rede aprendeu;


- Visualização de mapas de calor ou de ativação de classes $\to$ útil para entender que parte de uma imagem é identificada como sendo de uma dada classe, de forma a permitir localizar objetos nas imagens.

In [ ]:
rna.variables[0]
f0 = rna.variables[0].numpy()
print(f0.shape)
print(f0[:,:,0,3])

### 6.1 Visualização das saídas das camadas

A visualização das ativações das camadas convolucionais intermediárias de uma RNA consiste em mostrar os mapas de características que são produzidas pelos filtros para uma dada entrada da rede.

Essa visualização permite verificar como uma dada entrada é decomposta pelos diversos filtros aprendidos pela RNA.

A saída de cada camada convolucional é um tensor 3D, onde se tem um número de “imagens” igual ao número de canais (filtros) da camada.

Cada canal (filtro) de um camada convolucional codifica de forma independente características diferentes de forma que a melhor forma de visualização é fazer o gráfico da saída de cada filtro como se fosse uma imagem 2D.

Para fazer isso, temos que realizar alguns passos:

1. O primeiro passo é carregar uma RNA já treinada;


2. Gerar um modelo do Keras que permite múltiplas saídas;


3. Obter uma imagem de teste $\to$ obviamente que essa imagem deve ser processada da mesma forma como foram processadas as imagens usadas no treinamento.


4. Executar a nova RNA em modo de predição;


5. Fazer gráficos das sáidas.


### Obter modelo de RN já treinada

Como acabamos de treinar uma RN, vamos usá-la para essa visualização. Copiando-a com outro nome.

In [ ]:
rna_vis = tf.keras.models.clone_model(rna)
rna_vis.summary()

### Gerar modelo com múltiplas saídas

Para visualizar as saídas das camadas de uma RNA deve-se criar um modelo Keras que recebe uma imagem como entrada e gera como saída as ativações das camadas que se deseja visualizar $\to$ a classe de modelos Funcional do Keras permite construir redes dess tipo.

Esse tipo de modelo é equivalente ao modelo sequencial de uma RNA, mas pode ter múltiplas entradas e múltiplas saídas.

Observa-se que no caso geral uma RNA pode ter qualquer número de entradas e saídas.


Esse tipo de modelo é criado usando dois argumentos $\to$ uma lista de tensores de entrada e uma lista de tensores de saída.

In [ ]:
# Importa classe de modelos do keras
from tensorflow.keras import models

# Define as saídas como sendo as ativações das 4 primeiras camadas da RNA
camadas_saidas = [layer.output for layer in rna.layers[:4]]
#camadas_saidas = ['conv2d_2', 'conv2d_3']

# Cria o modelo que retorna as ativações das camadas, dada uma entrada
rna_ativacoes = models.Model(inputs=rna.inputs, outputs=camadas_saidas)

rna_ativacoes.summary()

- Quando esse modelo recebe uma imagem de entrada, ele retorna as ativações das camadas da RNA original.


- No caso dessa RNA tem-se uma entrada e quatro saídas (uma saída para cada conjunto de ativações de uma camada).

### Obter uma imagem de teste

A imagem usada como entrada dessa nova RNA deve ser um tensor de mesmo tamanho que o usado na RNA original.

No caso de ser uma imagem colorida, ela tem 3 eixos (altura, largura, cor) e o tensor de entrada da RNA tem 4 eixos (exemplo, altura, largura, cor) $\to$ portanto, deve-se incluir um quarto eixo na imagem antes dela ser usada como entrada da RNA.

In [ ]:
# Escolhe imagem do conjunto de teste
index = 0
imagem = x_test[index]
plt.imshow(np.squeeze(imagem), cmap='gray')
print ("É uma figura do número " + str(y_test[index]))
plt.show()

# Inclui eixo dos exemplos na imagem
imagem = np.expand_dims(imagem, 0)
print('Dimensão da imagem com o eixo dos exemplos =', imagem.shape)

### Prever saídas da RNA

O código abaixo executa a `rna_ativacoes` e separa as ativações das várias camadas em  tensores.

In [ ]:
# Calcula saídas da RNA (saídas das 4 primeiras camadas)
ativacoes = rna_ativacoes.predict(imagem)

# Separa ativações das camadas em tensores
first_layer = ativacoes[0]
sec_layer = ativacoes[1]
trd_layer = ativacoes[2]
four_layer = ativacoes[3]

print("Dimensão do tensor de saída da primeira camada convolucional =", first_layer.shape)
print("Dimensão do tensor de saída da segunda camada maxpooling =", sec_layer.shape)
print("Dimensão do tensor de saída da terceira camada convolucional =", trd_layer.shape)
print("Dimensão do tensor de saída da quarta camada pooling =", four_layer.shape)

- A saída da 1ª camada convolucional é um mapa de características de dimensão 24x24 com 8 canais.


- A saída da 2ª camada convolucional é um mapa de características de dimensão 12x12 com 8 canais.


- A saída da 3ª camada convolucional é um mapa de características de dimensão 8x8 com 16 canais.


- A saída da 4ª camada convolucional é um mapa de características de dimensão 4x4 com 16 canais.

### Visualização das ativações

As ativações do primeiro filro da 1ª camada convolucional são visualizadas da seguinte forma.

In [ ]:
index = 7
plt.matshow(first_layer[0,:,:,index], cmap='gray')

Visulização das saídas de todos os filtros de todas as camadas convolucionais e max-polling para a imagem de entrada.

In [ ]:
# Visualização de todos os canais das saídas das camadas convolucionais selecionadas
layer_names = []

for layer in rna.layers[:4]:
    layer_names.append(layer.name)

images_per_row = 8

for layer_name, layer_activation in zip(layer_names, ativacoes):
    n_features = layer_activation.shape[-1]
    size = layer_activation.shape[1]
    n_cols = n_features // images_per_row
    display_grid = np.zeros((size * n_cols, images_per_row * size))
    for col in range(n_cols):
        for row in range(images_per_row):
            channel_image = layer_activation[0,:,:,col * images_per_row + row]
            channel_image -= channel_image.mean()
            channel_image /= channel_image.std()
            channel_image *= 64
            channel_image += 128
            channel_image = np.clip(channel_image, 0, 255).astype('uint8')
            display_grid[col * size : (col + 1) * size,row * size : (row + 1) * size] = channel_image
    scale = 1 / size
    plt.figure(figsize=(scale * display_grid.shape[1],scale * display_grid.shape[0]))
    plt.title(layer_name)
    plt.grid(False)
    plt.imshow(display_grid, aspect='auto', cmap='gray')

### Conclusões

Em geral as primeiras camadas de uma RNA convolucional agem como uma coleção de detectores de vários tipos de bordas.

Nas primeiras camadas a ativações contém quase toda a informação presente na imagem original.
Na medida que avançamos para dentro da rede, as ativações se tornam mais abstratas e com menor significado visual e começam a codificar características de alto nível.

Características de níveis mais alto contém menos informação visual e mais informações relacionadas com a tarefa a ser realizada.

A não ativação de filtros aumenta com a profundidade da camada $\to$ na 1ª camada praticamente todos os filtros são ativados, mas nas camadas mais profundas menos filtros ficam ativos.

Quando um filtro não é ativado por uma imagem significa que o padrão codificado por aquele filtro não está presente naquela imagem.

Uma característica importante das RNAs convolucionais deep learning é que as características aprendidas pelas suas camadas se tornam cada vez mais abstratas com a profundidade da camada.
As ativações de camadas mais profundas contém menos informação visual e mais informação sobre a tarefa a ser realizada.

Uma RNA deep learning age efetivamente como um destilador de informação, onde dados brutos são repetidamente transformados de forma que informações irrelevantes são descartadas e informações importantes são ressaltadas e refinadas.

A forma que uma RNA convolucional deep learning opera é análoga à forma como os seres humanos e animais percebem o mundo $\to$ após observar uma cena por alguns segundos uma pessoa pode lembrar quais objetos estavam presentes (bicicleta, árvore, carro etc), mas não é capaz de lembrar de aspectos específicos desses objetos.

O cérebro humano é treinado para abstrair as imagens vistas e  transformá-las em conceitos visual de alto nível, descartando detalhes visuais irrelevantes.

No próximo trabalho você irá verificar pessoalmente essas conclusões.